In [1]:
!pip install crewai crewai-tools langchain-community python-dotenv

In [2]:
import os
from dotenv import load_dotenv,find_dotenv

def load_env():
    load_dotenv(find_dotenv())

def get_openai_api_key():
    load_env()
    openai_api_key=os.getenv("OPENAI_API_KEY")
    return openai_api_key

# def get_serper_api_key():
#     load_env()
#     openai_api_key=os.getenv("SERPER_API_KEY")
#     return openai_api_key
    

In [3]:
#warning controll
import warnings
warnings.filterwarnings('ignore')


In [4]:
# import libraries,apis,llm
from crewai import Agent,Task,Crew

In [5]:
import os
openai_api_key=get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"]="gpt-5.6-luna"
# os.environ["SERPER_API_KEY"]=get_serper_api_key()


In [6]:
# crew ai tools
from crewai_tools import(
    FileReadTool,
    ScrapeWebsiteTool,
    MDXSearchTool,
    SerperDevTool
)

search_tool=SerperDevTool()
scrape_tool=ScrapeWebsiteTool()
read_resume=FileReadTool(file_path="./rohit_cv.md")
semantic_search_resume=MDXSearchTool(file_path="./rohit_cv.md")



In [7]:
from IPython.display import Markdown,display
display(Markdown('./rohit_cv.md'))


# Rohit Kumar 

**Arrah, Bihar |** kumarrohit23502@gmail.com | +91 9199468225 LinkedIn | GitHub 

## **Professional Summary** 

Aspiring Data Analyst with knowledge of Python, MySQL, Power BI, Tableau, Seaborn, Plotly, and data visualization. Skilled in data cleaning, analysis, dashboard creation, SQL querying, and transforming raw data into actionable insights. Strong analytical mindset with a Computer Applications background and software development experience. 

## **Technical Skills** 

- Programming: Python 

- Databases: MySQL 

- Data Visualization: Power BI, Tableau, Seaborn, Plotly, Matplotlib 

- Data Analysis: Data Cleaning, Data Transformation, Exploratory Data Analysis (EDA) 

- Tools: Jupyter Notebook, Git, GitHub, Excel 

- Concepts: SQL Queries, Data Modeling, Dashboard Development, Reporting 

## **Experience** 

Full Stack Intern | Sihari Labs Pvt. Ltd. | Jul 2025 – Jan 2026 

- Worked with databases and APIs to manage and analyze application data. 

- Improved database structures and optimized queries for better performance. 

- Collaborated on data-driven features and reporting requirements. 

## **Projects** 

Sales Dashboard Analysis (Power BI) 

- Built interactive dashboards to track sales trends, KPIs, and business performance. - Created visual reports using charts, filters, and drill-down analysis. 

SQL Data Analysis Project 

- Wrote complex SQL queries for data extraction, aggregation, and reporting. 

- Analyzed customer and transaction datasets to identify patterns and insights. 

## **Education** 

Bachelor of Computer Applications (BCA) Cimage Professional College | 2021 – 2024 | CGPA: 7.51 



In [8]:
# Creating Agents
# Agent 1:- researcher
researcher=Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on" "job posting to help job applicants",
    tools=[scrape_tool,search_tool],
    verbose=True,
    backstory=(
        "As a job researcher, your progress in"
        "navigating and extracting critical"
        "information from job positings is unmatched."
        "Your skills help pinpoint the necessary"
        "qualifications and skills sought"
        "by employers, forming the foundation for"
        "effective application tailoring"
    )
)

In [9]:
# Agent 2:- profile agent

profiler=Agent(
    role="Personal Profiles for Engineers",
    goal="Do increditble research on job applications"
    "to help them stand out in the job market",
    tool=[scrape_tool,search_tool,read_resume,semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical process, you dissect"
        "and synthesize information"
        "from diverse sources to craft comprehension"
       "personal and professional profiles laying the"
       "groundwork for personalized resources enhancements" 
    )
)

In [10]:
# Agent 3:- resume strategist (Tailoring the resumes)
resume_strategist=Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a"
    "resume stand out in the job market",
    tools=[scrape_tool,search_tool,read_resume,semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you"
        "exceed at refining resumes to highlight the most"
        "relevant skills and experiences ensuring they"
        "reonate perfectly with the job's requirements"
    )
)

In [11]:
#Agent 4 :- Interview Preparer
interview_preparer=Agent(
    role="Engineering Interview Preparer.",
    goal="Create  interview questions and talking points"
    "based on the resume and job requirement",
    tools=[scrape_tool,search_tool,read_resume,semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of"
        "interviews. With your ability to formulate key questions"
        "and talking points, you prepare candidates for seekless"
        "ensuring they can confidently address all aspects of the"
        "job they are applying for."
    )
)

In [12]:
# creating tasks
research_task=Task(
    description=(
        "Analyse  the job posting url provided ({job_posting_url})"
        "to execute key skills,experiences and qualifications"
        "required. Use the tools together,content and identify"
        "and categorize the requirement"
    ),
    expected_output=(
        "A structured list of job requirements,including necessary"
        "skills,qualifications and experiences"
    ),
    agent=researcher,
    async_execution=True
)

In [13]:
# creating tasks
profile_task=Task(
    description=(
        "Compile a detailed personal and professional profile using the personal write_up"
        "({personal_writtenup}). If a Github URL is provided,extract project insights as well"
        "Otherwise, focus on synthesizing information from the personal summary alone."
    ),
    expected_output=(
        "A comprehension profile document that includes skills,experience,intrests and communication style."
        "If github or linkedin data is available included. Highlights from open-source contributors and projects."
    ),
    agent=profiler,
    async_execution=True
)

In [14]:
# creating tasks
resume_strategy_task=Task(
    description=(
        "Using the profile and job requirements obtained from."
        "previous tasks tailor  the resume to highlight the most"
        "relevant areas. Employ tools to adjust and enhance the"
        "resume content.Make sure this is the best resume even but"
        "don't make up any information update every selection"
        "including the initial summary work experience,skills"
        "and education.All to better  reflect the candidates"
        "abilities and how it matches the job postings"
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's"
        "qualifications and experiences relevant to the job."
        ),
    output_file="tailored_resume.md",
    context=[research_task,profile_task],

    agent=resume_strategist,

)

In [15]:
# interview preparation task
interview_preparation_task=Task(
    description=(
        "Based on the customized resume and the job requirements,create a focused set of interview questions"
        "and supporting talking points.Use the provided tools to ensure relevance and alignment with the role."
        "The materials should help the candidate clearly articulate how their skills and experiences relate to the job"
        "and guide them in emphasizing key resume highlight during the interview"
    ),
    expected_output=(
        "A structured document featuring suggested interview questions and talking points"
        "to help the candidate effectively prepare for a first-round interview."
    ),
    output_file="interview_questions.md",
    context=[research_task,profile_task,resume_strategy_task],
    agent=interview_preparer
)

In [16]:
#  creating the crew
job_application_crew=Crew(
    agents=[
        researcher,
        profiler,
        resume_strategist,
        interview_preparer
    ],
    tasks=[
        research_task,
        profile_task,
        resume_strategy_task,
        interview_preparation_task
    ],
    verbose=True
)

In [17]:
job_application_inputs = {
    "job_posting_url": "https://cornerstone.csod.com/ux/ats/careersite/2/home/requisition/11395?c=cornerstone&sq=Junior%20Engineer&country=in",
    "github_url": "https://github.com/yourusername",
    "personal_writtenup": (
        "Rohit is a seasoned Software Engineer Leader with over 7 years of experience. "
        "Highly skilled in leading global engineering teams and building scalable, high-impact technology solutions. "
        "Rohit has successfully founded and led startups, demonstrating a strong track record of innovation, execution, "
        "and cross-functional collaboration. He is ideally suited for senior leadership roles requiring technical excellence, vision, and operational expertise."
    )
}

In [18]:
result=await job_application_crew.kickoff_async(inputs=job_application_inputs)
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.7                                                                                        │
│  Latest version:  1.15.9                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 23dce871-d5a9-45ab-8b61-4c9de2837cb7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse  the job posting url provided                                                                    │
│  (https://cornerstone.csod.com/ux/ats/careersite/2/home/requisition/11395?c=cornerstone&sq=Junior%20Engineer&c  │
│  ountry=in)to execute key skills,experiences and qualificationsrequired. Use the tools together,content and     │
│  identifyand categorize the requirement                                                                         │
│  ID: 0976f533-34aa-4635-b530-82467df45020                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Compile a detailed personal and professional profile using the personal write_up(Rohit is a seasoned     │
│  Software Engineer Leader with over 7 years of experience. Highly skilled in leading global engineering teams   │
│  and building scalable, high-impact technology solutions. Rohit has successfully founded and led startups,      │
│  demonstrating a strong track record of innovation, execution, and cross-functional collaboration. He is        │
│  ideally suited for senior leadership roles requiring technical excellence, vision, and operational             │
│  expertise.). If a Github URL is provided,extract project insights as wellOtherwise, focus on synthesizing      │
│  information from the personal summary alone.                                                                   │
│  ID: a23a47dd-c9d9-4a06-ab05-98877d2d8c1d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Profiles for Engineers                                                                         │
│                                                                                                                 │
│  Task: Compile a detailed personal and professional profile using the personal write_up(Rohit is a seasoned     │
│  Software Engineer Leader with over 7 years of experience. Highly skilled in leading global engineering teams   │
│  and building scalable, high-impact technology solutions. Rohit has successfully founded and led startups,      │
│  demonstrating a strong track record of innovation, execution, and cross-functional collaboration. He is        │
│  ideally suited for senior leadership roles requiring technical excellence, vision, and operational             │
│  expertise.). If a Github URL is provided,extract project insights as wellOtherwise, focus on synthesizing      │
│  information from the personal summary alone.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Engineering Interview Preparer.                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # First-Round Interview Preparation: Junior Engineer                                                           │
│                                                                                                                 │
│  ## 1. Candidate Positioning                                                                                    │
│                                                                                                                 │
│  ### Recommended opening introduction                                                                           │
│                                                                                                                 │
│  > I’m a Junior Software Engineer with a Bachelor of Computer Applications and hands-on experience as a Full    │
│  Stack Intern at Sihari Labs. My primary strengths are Python, SQL, MySQL, API integration, application data    │
│  management, and data analysis. During my internship, I contributed to full-stack development, improved         │
│  database structures, optimized SQL queries, and supported data-driven features and reporting requirements. I   │
│  have also built projects using Power BI and SQL to analyze business data and present useful insights. I’m now  │
│  looking for an opportunity where I can apply my software development fundamentals, continue strengthening my   │
│  backend and database skills, and contribute effectively while learning from an experienced engineering team.   │
│                                                                                                                 │
│  ### Core message to communicate                                                                                │
│                                                                                                                 │
│  Emphasize that you are:                                                                                        │
│                                                                                                                 │
│  - A junior engineer with practical internship experience, not only academic knowledge.                         │
│  - Comfortable working with Python, SQL, MySQL, APIs, and application data.                                     │
│  - Interested in software development and backend/database engineering.                                         │
│  - Able to analyze, clean, transform, and present data.                                                         │
│  - Familiar with Git, GitHub, Jupyter Notebook, and development workflows.                                      │
│  - A fast learner who can adapt to unfamiliar technologies.                                                     │
│  - Detail-oriented and comfortable debugging data or application issues.                                        │
│  - Able to collaborate with a team and communicate technical findings clearly.                                  │
│                                                                                                                 │
│  ### Important clarification                                                                                    │
│                                                                                                                 │
│  The supplied job posting only identifies the role as *

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the customized resume and the job requirements,create a focused set of interview questionsand   │
│  supporting talking points.Use the provided tools to ensure relevance and alignment with the role.The           │
│  materials should help the candidate clearly articulate how their skills and experiences relate to the joband   │
│  guide them in emphasizing key resume highlight during the interview                                            │
│  Agent: Engineering Interview Preparer.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 23dce871-d5a9-45ab-8b61-4c9de2837cb7                                                                       │
│  Final Output: # First-Round Interview Preparation: Junior Engineer                                             │
│                                                                                                                 │
│  ## 1. Candidate Positioning                                                                                    │
│                                                                                                                 │
│  ### Recommended opening introduction                                                                           │
│                                                                                                                 │
│  > I’m a Junior Software Engineer with a Bachelor of Computer Applications and hands-on experience as a Full    │
│  Stack Intern at Sihari Labs. My primary strengths are Python, SQL, MySQL, API integration, application data    │
│  management, and data analysis. During my internship, I contributed to full-stack development, improved         │
│  database structures, optimized SQL queries, and supported data-driven features and reporting requirements. I   │
│  have also built projects using Power BI and SQL to analyze business data and present useful insights. I’m now  │
│  looking for an opportunity where I can apply my software development fundamentals, continue strengthening my   │
│  backend and database skills, and contribute effectively while learning from an experienced engineering team.   │
│                                                                                                                 │
│  ### Core message to communicate                                                                                │
│                                                                                                                 │
│  Emphasize that you are:                                                                                        │
│                                                                                                                 │
│  - A junior engineer with practical internship experience, not only academic knowledge.                         │
│  - Comfortable working with Python, SQL, MySQL, APIs, and application data.                                     │
│  - Interested in software development and backend/database engineering.                                         │
│  - Able to analyze, clean, transform, and present data.                                                         │
│  - Familiar with Git, GitHub, Jupyter Notebook, and development workflows.                                      │
│  - A fast learner who can adapt to unfamiliar technologies.                                                     │
│  - Detail-oriented and comfortable debugging data or application issues.                                        │
│  - Able to collaborate with a team and communicate technical findings clearly.                                  │
│                                                                                                                 │
│  ### Important clarification                                                                                    │
│                                                                                                                 │
│  The supplied job posting only identifies the role as 

# First-Round Interview Preparation: Junior Engineer

## 1. Candidate Positioning

### Recommended opening introduction

> I’m a Junior Software Engineer with a Bachelor of Computer Applications and hands-on experience as a Full Stack Intern at Sihari Labs. My primary strengths are Python, SQL, MySQL, API integration, application data management, and data analysis. During my internship, I contributed to full-stack development, improved database structures, optimized SQL queries, and supported data-driven features and reporting requirements. I have also built projects using Power BI and SQL to analyze business data and present useful insights. I’m now looking for an opportunity where I can apply my software development fundamentals, continue strengthening my backend and database skills, and contribute effectively while learning from an experienced engineering team.

### Core message to communicate

Emphasize that you are:

- A junior engineer with practical internship experience, not on

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯